# Growth model and Monte Carlo design pathways

Inputs:
- `input/scenario_inputs.xlsx`: 2030 cumulative storage, storage resources and prescribed 2050 targets by jurisdiction

## Settings

In [ ]:
import hashlib
import math
from pathlib import Path

import numpy as np
import pandas as pd
import openpyxl
from scipy.special import expit

ROOT = Path.cwd().resolve()
if not (ROOT / 'input').is_dir() and (ROOT.parent / 'input').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'input').is_dir(), 'Run from the Global Study folder or its code folder.'
INPUT_FILE = ROOT / 'input' / 'scenario_inputs.xlsx'
OUTPUT_DIR = ROOT / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = ['reference', 'minimum', 'maximum', 'growth10', 'us1gt', 'policy']
MODELS = ['Logistic', 'Gompertz']
SHEETS = {'UK': 'UK', 'US': 'US', 'EU': 'EU without UK', 'China': 'China', 'Middle East': 'Middle East',
          'Australia': 'Australia', 'Canada': 'Canada', 'Indonesia': 'Indonesia', 'Thailand': 'Thailand',
          'Brazil': 'Brazil'}

START_YEAR, END_YEAR = 2030, 2180
T0 = float(START_YEAR)
N_DRAWS = 40_000
G_MIN = 0.01
TARGET_MIN = 0.001
TP_ACCEPT_MIN = 2050.0
ANCHOR_QUANTILE = 0.95
FIT_RTOL = 1e-10
SCAN_NODES = 257
BISECTION_STEPS = 55
BATCH = 2048

PHI = (3 + np.sqrt(5.0)) / 2
MODEL_CONSTANTS = {'Logistic': (1 / (3 + np.sqrt(3.0)), np.log(2 + np.sqrt(3.0))),
                   'Gompertz': (np.exp(-PHI), np.log(PHI))}

def pool_seed(scenario, country):
    # fixed seed per pool: first 4 bytes of SHA-256('scenario|region')
    return int.from_bytes(hashlib.sha256(f'{scenario}|{country}'.encode()).digest()[:4], 'big')

## Inputs

In [ ]:
def load_inputs(path):
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    capacity = {str(r[0]).strip(): float(r[1])
                for r in wb['Geological Storage Capacity'].iter_rows(min_row=2, values_only=True) if r[0] is not None}
    ws = wb['IPCCreference']
    col = [str(c.value).strip() for c in ws[1]].index('Published storage target [Gt/yr]')
    policy = {str(r[0]).strip(): float(r[col]) if isinstance(r[col], (int, float)) else np.nan
              for r in ws.iter_rows(min_row=2, values_only=True) if r[0] is not None}
    rows = []
    for country, sheet in SHEETS.items():
        r2030 = [r for r in wb[sheet].iter_rows(min_col=1, max_col=4, values_only=True) if r[0] == START_YEAR]
        assert len(r2030) == 1, sheet
        rows.append(dict(country=country, S0_Gt=float(r2030[0][3]), C_reference_Gt=capacity[country],
                         policy_target_Gt_yr=policy.get(country, np.nan)))
    wb.close()
    df = pd.DataFrame(rows)
    assert (df[['S0_Gt', 'C_reference_Gt']].to_numpy() > 0).all()
    return df

country_inputs = load_inputs(INPUT_FILE)
country_inputs.round(6)

## Growth functions and exact calibration

In [ ]:
def calibrated_profile(model, S0, g, x):
    alpha, a = MODEL_CONSTANTS[model]
    x, g = np.broadcast_arrays(np.asarray(x, float), np.asarray(g, float))
    if np.any(x < 0) or np.any(g <= 0):
        raise ValueError('Profile requires x >= 0 and g > 0.')
    lam = np.log1p(g)
    small = x < 1e-4
    safe_x = np.where(small, 1.0, x)
    C = S0 / alpha * np.exp(x)
    if model == 'Logistic':
        p = 1 - alpha
        delta = np.log1p(np.expm1(x) / p)
        d1, d2 = 1 / p, -alpha / p**2
        d3, d4 = alpha * (1 + alpha) / p**3, -alpha * (1 + 4 * alpha + alpha**2) / p**4
        r = lam * np.where(small, d1 + d2 * x / 2 + d3 * x * x / 6 + d4 * x**3 / 24, delta / safe_x)
        w = a + delta - 20 * r
        q = C * r * expit(-w) * expit(w)
    elif model == 'Gompertz':
        delta = np.log1p(x / PHI)
        r = lam * np.where(small, 1 / PHI - x / (2 * PHI**2) + x * x / (3 * PHI**3) - x**3 / (4 * PHI**4),
                           delta / safe_x)
        u = (PHI + x) * np.exp(-20 * r)
        q = C * r * u * np.exp(-u)
    else:
        raise ValueError(model)
    return C, r, T0 + x / lam, T0 + (a + delta) / r, q

def anchored_curve_values(model, C, r, S0, year):
    C, r, S0 = np.asarray(C), np.asarray(r), np.asarray(S0)
    tau = np.asarray(year) - T0
    if model == 'Logistic':
        z = np.log((C - S0) / S0) - r * tau
        return C * expit(-z), C * r * expit(-z) * expit(z)
    if model == 'Gompertz':
        u = np.log(C / S0) * np.exp(-r * tau)
        return C * np.exp(-u), C * r * u * np.exp(-u)
    raise ValueError(model)

def realised_cagr(model, C, r, S0):
    alpha, _ = MODEL_CONSTANTS[model]
    x = np.log(alpha * np.asarray(C) / S0)
    if model == 'Logistic':
        delta, limit = np.log1p(np.expm1(x) / (1 - alpha)), 1 / (1 - alpha)
    elif model == 'Gompertz':
        delta, limit = np.log1p(x / PHI), 1 / PHI
    else:
        raise ValueError(model)
    delta_over_x = np.divide(delta, x, out=np.full_like(x, limit), where=x != 0)
    return np.expm1(np.asarray(r) / delta_over_x)

def bisect_vectors(fn, lo, hi, iterations=BISECTION_STEPS):
    lo, hi = np.array(lo, float, copy=True), np.array(hi, float, copy=True)
    fl, fh = fn(lo), fn(hi)
    if np.any(fl * fh > 0):
        raise RuntimeError('Unbracketed vector root.')
    left_zero, right_zero = fl == 0, fh == 0
    fixed = np.where(left_zero, lo, hi)
    exact = left_zero | right_zero
    for _ in range(iterations):
        mid = (lo + hi) / 2
        fm = fn(mid)
        same = np.signbit(fm) == np.signbit(fl)
        lo = np.where(same, mid, lo)
        fl = np.where(same, fm, fl)
        hi = np.where(same, hi, mid)
    return np.where(exact, fixed, (lo + hi) / 2)

## Scenarios and proposal ranges

In [ ]:
def attainable_upper(model, S0, C_cap, g_cap, n_g=401, n_x=2001):
    xmax = float(np.log(MODEL_CONSTANTS[model][0] * C_cap / S0))
    xs = np.linspace(0.0, xmax, n_x)

    def best_over_x(gs):
        return calibrated_profile(model, S0, np.asarray(gs, float)[:, None], xs[None, :])[4].max(axis=1)

    gs = np.linspace(G_MIN, g_cap, n_g)
    q_best = best_over_x(gs)
    k = int(np.argmax(q_best))
    lo, hi = gs[max(k - 1, 0)], gs[min(k + 1, n_g - 1)]
    for _ in range(60):
        g1, g2 = hi - (hi - lo) * 0.6180339887498949, lo + (hi - lo) * 0.6180339887498949
        if best_over_x([g1])[0] < best_over_x([g2])[0]:
            lo = g1
        else:
            hi = g2
    return max(float(q_best.max()), float(best_over_x([(lo + hi) / 2])[0]))

def scenario_table(base):
    rows = []
    for scenario in SCENARIOS:
        for b in base.to_dict('records'):
            country = b['country']
            factor = 0.1 if scenario == 'minimum' else 10.0 if scenario == 'maximum' else 1.0
            cap = b['C_reference_Gt'] * factor
            gcap = 0.1 if scenario in ['minimum', 'growth10'] else (0.25 if country == 'China' else 0.2)
            target = 1.0 if scenario == 'us1gt' and country == 'US' else np.nan
            if scenario == 'policy':
                target = b['policy_target_Gt_yr']
            per_form = {m: attainable_upper(m, b['S0_Gt'], cap, gcap) for m in MODELS}
            upper = max(per_form.values())
            assert np.isfinite(upper) and upper > TARGET_MIN, (scenario, country)
            rows.append(dict(scenario=scenario, country=country, S0_Gt=b['S0_Gt'], C_cap_Gt=cap, g_cap=gcap,
                             target_type='fixed' if np.isfinite(target) else 'uniform',
                             target_fixed_Gt_yr=target, target_upper_Gt_yr=upper,
                             target_upper_Logistic_Gt_yr=per_form['Logistic'],
                             target_upper_Gompertz_Gt_yr=per_form['Gompertz'],
                             seed=pool_seed(scenario, country)))
    return pd.DataFrame(rows)

scenario_inputs = scenario_table(country_inputs)
scenario_inputs[['scenario', 'country', 'C_cap_Gt', 'g_cap', 'target_type', 'target_fixed_Gt_yr',
                 'target_upper_Logistic_Gt_yr', 'target_upper_Gompertz_Gt_yr']].round(4)

## Monte Carlo

In [ ]:
def proposal_arrays(row, n=N_DRAWS):
    rng = np.random.default_rng(int(row['seed']))
    c_low = row['S0_Gt'] / MODEL_CONSTANTS['Logistic'][0]
    u = rng.uniform(0.0, 1.0, n)
    Cs = np.exp(np.log(c_low) + u * (np.log(row['C_cap_Gt']) - np.log(c_low)))
    targets = (rng.uniform(TARGET_MIN, row['target_upper_Gt_yr'], n)
               if row['target_type'] == 'uniform' else np.full(n, row['target_fixed_Gt_yr']))
    return Cs, targets

def solve_g(model, S0, C_cap, g_cap, Cs, targets):
    alpha, _ = MODEL_CONSTANTS[model]
    n = len(Cs)
    x = np.log(alpha * Cs / S0)
    in_domain = (x > 0) & (Cs < C_cap)
    xs = np.where(in_domain, x, 1.0)
    ggrid = np.linspace(G_MIN, g_cap, SCAN_NODES)
    above = calibrated_profile(model, S0, ggrid[None, :], xs[:, None])[4] >= targets[:, None]
    k = np.argmax(above, axis=1)
    has = above.any(axis=1) & (k > 0) & in_domain
    root = np.full(n, np.nan)
    ii = np.flatnonzero(has)
    if len(ii):
        root[ii] = bisect_vectors(lambda gg: calibrated_profile(model, S0, gg, xs[ii])[4] - targets[ii],
                                  ggrid[k[ii] - 1], ggrid[k[ii]])
    g = np.where(np.isfinite(root), root, G_MIN)
    C, r, tn, tp, _ = calibrated_profile(model, S0, g, xs)
    rate_2050 = anchored_curve_values(model, C, r, S0, 2050.0)[1]
    stock_2030 = anchored_curve_values(model, C, r, S0, T0)[0]
    verified = (np.isfinite(root) & (abs(rate_2050 - targets) / targets <= FIT_RTOL)
                & (abs(stock_2030 - S0) / S0 <= FIT_RTOL)
                & (abs(realised_cagr(model, C, r, S0) - g) / g <= FIT_RTOL))
    status = np.select([~in_domain, ~np.isfinite(root), ~verified, ~(tp > TP_ACCEPT_MIN)],
                       ['resource_boundary', 'no_target_intersection', 'numerical_fit_failure',
                        'peak_year_constraint'], default='retained')
    return pd.DataFrame(dict(draw_id=np.arange(n), g_draw=g, target_2050_Gt_yr=targets, C_Gt=C, r_per_yr=r,
                             tn=tn, tp=tp, rate_2050_Gt_yr=rate_2050, status=status))

def fit_pool(row, model):
    Cs, targets = proposal_arrays(row)
    parts = []
    for start in range(0, N_DRAWS, BATCH):
        part = solve_g(model, row['S0_Gt'], row['C_cap_Gt'], row['g_cap'],
                       Cs[start:start + BATCH], targets[start:start + BATCH])
        part['draw_id'] += start
        parts.append(part)
    return pd.concat(parts, ignore_index=True)

In [ ]:
REJECTIONS = ['no_target_intersection', 'numerical_fit_failure', 'resource_boundary', 'peak_year_constraint']
pools, mc_stats = {}, []
for row in scenario_inputs.to_dict('records'):
    for model in MODELS:
        fits = fit_pool(row, model)
        kept = fits.loc[fits.status == 'retained', ['draw_id', 'g_draw', 'target_2050_Gt_yr', 'C_Gt', 'r_per_yr',
                                                     'tn', 'tp', 'rate_2050_Gt_yr']].copy()
        assert kept.g_draw.between(G_MIN, row['g_cap']).all() and (kept.tp > TP_ACCEPT_MIN).all()
        for year in (2050, 2100, END_YEAR):
            stock, rate = anchored_curve_values(model, kept.C_Gt.to_numpy(), kept.r_per_yr.to_numpy(),
                                                row['S0_Gt'], float(year))
            kept[f'cumulative_{year}_Gt'] = stock
            if year != 2050:
                kept[f'rate_{year}_Gt_yr'] = rate
        pools[(row['scenario'], row['country'], model)] = kept
        st = dict(scenario=row['scenario'], country=row['country'], model=model, pool_retained=len(kept),
                  pool_peak_ge_2100=int((kept.tp >= 2100).sum()),
                  pool_peak_after_end_year=int((kept.tp > END_YEAR).sum()))
        st.update({f'rejected_{s}': int((fits.status == s).sum()) for s in REJECTIONS})
        for col in ['rate_2050_Gt_yr', 'C_Gt', 'tp']:
            for label, q in [('p05', .05), ('p50', .5), ('p95', .95)]:
                st[f'pool_{col}_{label}'] = float(kept[col].quantile(q)) if len(kept) else np.nan
        mc_stats.append(st)
    print(f"{row['scenario']:9s} {row['country']:12s} accepted: "
          + ' / '.join(f"{m} {len(pools[(row['scenario'], row['country'], m)]):,}" for m in MODELS))

mc_stats = pd.DataFrame(mc_stats)
assert (mc_stats.pool_retained >= 1000).all(), 'A pool has fewer than 1,000 accepted trajectories.'
samples = pd.concat([df.assign(scenario=s, country=c, model=m) for (s, c, m), df in pools.items()],
                    ignore_index=True)
samples.to_csv(OUTPUT_DIR / 'mc_samples.csv.gz', index=False, compression={'method': 'gzip', 'mtime': 0})
mc_stats.groupby(['scenario', 'model'], sort=False).pool_retained.sum().unstack()

## Design pathways

In [ ]:
def solve_central_g(model, S0, C, target, g_cap, n_nodes=2001):
    x = float(np.log(MODEL_CONSTANTS[model][0] * C / S0))
    f = lambda gg: calibrated_profile(model, S0, np.asarray(gg, float), x)[4] - target
    gs = np.linspace(G_MIN, g_cap, n_nodes)
    fv = f(gs)
    roots = list(gs[fv == 0])
    ids = np.flatnonzero(fv[:-1] * fv[1:] < 0)
    if len(ids):
        roots += list(bisect_vectors(f, gs[ids], gs[ids + 1]))
    return sorted(set(float(r) for r in roots))

central_rows = []
for row in scenario_inputs.to_dict('records'):
    key = (row['scenario'], row['country'])
    S0, C_sc, g_cap = row['S0_Gt'], row['C_cap_Gt'], row['g_cap']
    logistic = pools[key + ('Logistic',)].set_index('draw_id')
    joint_ids = logistic.index.intersection(pools[key + ('Gompertz',)].draw_id)
    joint_targets = logistic.loc[joint_ids, 'target_2050_Gt_yr'].to_numpy(float)
    n_joint = len(joint_targets)
    if row['target_type'] == 'fixed':
        anchor, anchor_rule = float(row['target_fixed_Gt_yr']), 'fixed_target'
    else:
        anchor = float(np.quantile(joint_targets, ANCHOR_QUANTILE, method='linear')) if n_joint else np.nan
        anchor_rule = 'p95_jointly_accepted_q2050'
    for model in MODELS:
        roots = solve_central_g(model, S0, C_sc, anchor, g_cap) if np.isfinite(anchor) else []
        rec = dict(scenario=key[0], country=key[1], model=model, S0_Gt=S0, C_scenario_Gt=C_sc, g_cap=g_cap,
                   target_type=row['target_type'], anchor_2050_Gt_yr=anchor, n_joint_draws=n_joint,
                   anchor_rule=anchor_rule, target_upper_form_Gt_yr=row[f'target_upper_{model}_Gt_yr'],
                   target_upper_union_Gt_yr=row['target_upper_Gt_yr'], n_g_solutions=len(roots))
        if roots:
            g_star = roots[0]
            x = np.log(MODEL_CONSTANTS[model][0] * C_sc / S0)
            C, r, tn, tp, _ = (float(v) for v in calibrated_profile(model, S0, g_star, x))
            stock, rate = {}, {}
            for y in (2030, 2050, 2100, END_YEAR):
                s, q = anchored_curve_values(model, C, r, S0, y)
                stock[y], rate[y] = float(s), float(q)
            assert abs(stock[2030] - S0) <= FIT_RTOL * S0 and abs(rate[2050] - anchor) <= FIT_RTOL * anchor
            rec.update(g_star=g_star, r_per_yr=r, tn=tn, tp=tp,
                       peak_after_2050=bool(tp > 2050), peak_before_2100=bool(tp < 2100),
                       peak_rate_Gt_yr=C * r / (4 if model == 'Logistic' else math.e),
                       **{f'rate_{y}_Gt_yr': rate[y] for y in (2030, 2050, 2100, END_YEAR)},
                       **{f'stock_{y}_Gt': stock[y] for y in (2050, 2100, END_YEAR)},
                       status='central' if tp > 2050 else 'peak_not_after_2050')
        else:
            rec.update(status='anchor_unreachable_at_C_scenario' if np.isfinite(anchor) else 'empty_joint_pool')
        central_rows.append(rec)

central = pd.DataFrame(central_rows).merge(mc_stats, on=['scenario', 'country', 'model'])
central.to_csv(OUTPUT_DIR / 'central_pathways.csv', index=False)
print(f'{len(central)} design pathways; peak after 2050: {int(central.peak_after_2050.sum())}; '
      f'peak before 2100: {int(central.peak_before_2100.sum())}')
central[['scenario', 'country', 'model', 'anchor_2050_Gt_yr', 'g_star', 'tp', 'status']]